In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from scipy.signal import savgol_filter

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


def load_data():

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    return X, y, X_test


# -----------------------------
# Savitzky–Golay smoothing
# -----------------------------
def apply_savgol(X, X_test):

    print("Applying Savitzky–Golay smoothing...")

    X_smooth = savgol_filter(
        X,
        window_length=11,
        polyorder=2,
        deriv=0
    )

    X_test_smooth = savgol_filter(
        X_test,
        window_length=11,
        polyorder=2,
        deriv=0
    )

    return X_smooth, X_test_smooth


def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    return X_scaled, X_test_scaled


def cross_validate(X, y):

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    rmse_scores = []

    for train_idx, val_idx in kf.split(X):

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = ElasticNet(
            alpha=1.0,
            l1_ratio=0.7,
            max_iter=100000,
            tol=1e-3,
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmse_scores.append(rmse)

    print("CV RMSE:", np.mean(rmse_scores))


def train_and_predict(X, y, X_test):

    model = ElasticNet(
        alpha=1.0,
        l1_ratio=0.7,
        max_iter=100000,
        tol=1e-3,
        random_state=42
    )

    model.fit(X, y)

    preds = model.predict(X_test)

    print("Sample predictions:", preds[:10])

    return preds


def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    experiment_name = "exp07_savgol_smoothing_elasticnet_20260323"

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("Submission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = apply_savgol(X, X_test)

    X, X_test = scale_features(X, X_test)

    cross_validate(X, y)

    preds = train_and_predict(X, y, X_test)

    save_submission(test, preds)


if __name__ == "__main__":
    main()

Train shape: (1322, 1559)
Test shape: (550, 1558)
Applying Savitzky–Golay smoothing...
CV RMSE: 24.104770404345622
Sample predictions: [189.18511242 178.26857335 170.08410669 163.7292478  154.7649975
 144.40151706 135.02418352 127.61620549 121.24446845 115.97524167]
Submission saved to: ../submissions/exp07_savgol_smoothing_elasticnet_20260323.csv
    0           1
0  95  189.185112
1  96  178.268573
2  97  170.084107
3  98  163.729248
4  99  154.764998
